# Lab 03 · Prompt chaining with quality gates

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## What is prompt chaining?
A prompt chain breaks a task into smaller model calls. The output of one call becomes input to the next. This lets us inspect each handoff instead of asking for everything at once.

First, build a two-call chain for an everyday lunch. Then use the same idea for ShopEasy: propose candidate bounded contexts, compare boundary strategies, and record a decision.

The order is fixed by our code. The model does not decide which step to run next. The full lab makes five model calls: two for lunch and three for the architecture exercise.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Step 1 · A simple chain: lunch → ingredients
Imagine you want a lunch idea and then a shopping list.

1. Ask the model to suggest one lunch. Save its answer in `lunch`.
2. Put `lunch` into a second prompt and ask for three ingredients.

For example, if the first answer is “vegetable pasta,” the second prompt asks for ingredients for vegetable pasta. Your answer may differ. **Notice:** the second call only knows the lunch because we explicitly include it in the prompt. This is the same handoff you practiced in Lab 01, now used to complete a task in stages.

In [ ]:
lunch = ask("Suggest one simple vegetarian lunch. Name the dish only.")
print("1. Lunch:", lunch)

ingredients = ask(f"The lunch is: {lunch}\nList three ingredients, one per line.")
print("2. Ingredients:\n", ingredients)

## Next: apply the chain to ShopEasy

ShopEasy sells jackets. A customer places an order, stock is reserved, payment is authorized, and a parcel is shipped. Order, Inventory, and Shipping currently share tables and releases. The next example uses several model calls to explore possible boundaries.

**At a glance**

![Lab 03 at a glance: Brief, then Business capabilities (3 labels), then gate, then Boundary options (2 options · 3 criteria), then Trade-offs (+ / − table), then gate, then Decision record (Context · Decision · Consequence), then gate.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/03.svg)

## Think about the example

A technical-layer split places UI, business logic, and data access in different services. A bounded context defines where one business model and language apply; its deployment boundary still needs review.

**Concepts:** Event storming; Bounded contexts and ubiquitous language; Business capability vs technical-layer decomposition.

Use the scenario above to consider these questions before running the code:

1. Why can a technical-layer split create a distributed monolith?
2. Why is a bounded context a service candidate rather than an automatic deployment decision?

## The idea in context

A fixed prompt chain exposes intermediate results. Gates can catch a malformed handoff while leaving domain judgement for review.

A bounded context is a model boundary; validate it before choosing deployment boundaries.

| Key term | Meaning |
| --- | --- |
| **Workflow** | A sequence of steps fixed by code; the model does not choose what happens next. |
| **Decomposition** | Splitting a large task into smaller steps with clear inputs and outputs. |
| **Gate** | A plain-code check between two steps that stops a bad result from spreading. |
| **Decision record (ADR)** | A short note of an architecture decision: its context, the decision, and its consequence. |

## Pattern · Single-Path Planning

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Passive Goal Creator.*

| | |
| --- | --- |
| **Context** | Complex goals need several dependent steps, where each result informs the next. |
| **Problem** | One monolithic response cannot reliably manage dependencies and accumulated intermediate results. |
| **Solution** | Make one ordered plan of a few steps, execute each step with the accumulated context, then synthesize the outputs. |
| **Use when** | The work is naturally sequential; steps depend on earlier outputs; one reasonable path is enough. |
| **Trade-off** | Easy to follow, but an early planning mistake propagates through the whole run. |
| **Flow** | Plan → Step 1 → Step N → Synthesize |
| **Quality attributes** | Benefits: Predictability, Performance, Observability, Maintainability. Liabilities: Reliability, Latency, Usability, Adaptability. |
| **In this lab** | Events lead to candidate bounded contexts, then a comparison of boundary strategies and a short decision record. Gates check output structure; domain experts still validate the proposed boundaries. |

## Step 2 · Prepare the ShopEasy example
ShopEasy sells jackets. A customer places an order, stock is reserved, payment is authorized, and a parcel is shipped. Order, Inventory, and Shipping currently share tables and releases.

The next three calls follow **events → candidate contexts → trade-offs → decision record**. After each call, ordinary `if` statements check the reply before it is used in the next prompt. This cell only prepares the input data.

In [ ]:
import json

brief = ("ShopEasy sells jackets. A customer places an order; stock is reserved; payment is authorized; "
         "a parcel is shipped. Order, Inventory and Shipping currently share tables and releases.")
events = ["OrderPlaced", "StockReserved", "PaymentAuthorized", "ParcelShipped"]
options = ["business capability split", "technical layer split"]
criteria = ["cohesive business rules", "independent change", "private data ownership"]
print("Events:", events)
print("Options:", options)

## Step 3 · Call 1: suggest bounded contexts
**Input:** the brief and events. **Output:** `contexts`, a list for the next call.

Four events happen in checkout; we ask the model to group three of them by business meaning. Payment is owned by an external provider in this small example, so `PaymentAuthorized` needs no home here. The `if` statement is a quality gate: it checks the JSON list has three context labels, and it stops the chain otherwise. Read the labels themselves too.

In [ ]:
prompt = (f"Brief: {brief}\nEvents: {events}\nPropose exactly 3 business bounded contexts for "
          "OrderPlaced, StockReserved and ParcelShipped. Payment is supplied externally. "
          "Reply with a JSON list of context labels only, no code fences or explanation.")
reply = ask(prompt)
print("Contexts reply:", reply)
contexts = json.loads(reply)
if not isinstance(contexts, list) or len(contexts) != 3:
    raise ValueError("Quality gate stopped the chain: expected a JSON list of three context labels. Rerun this cell to ask again.")

## Step 4 · Call 2: compare boundary choices
**Input:** the brief and `contexts` from Step 3. **Output:** `tradeoffs`, a table for the final call.

Ask the model to compare the two options. Then use loops to check every option and criterion has a `+` or `-`. These marks are judgements, not measurements; inspect them before continuing.

In [ ]:
prompt = (f"Brief: {brief}\nContexts: {contexts}\nCompare {options} using {criteria}. "
          'JSON only: {"<option>": {"<criterion>": "+" or "-"}}. Marks are qualitative, not measured.')
reply = ask(prompt)
print("Trade-offs reply:", reply)
tradeoffs = json.loads(reply)
if not isinstance(tradeoffs, dict):
    raise ValueError("Quality gate stopped the chain: expected a JSON object.")
for option in options:
    marks = tradeoffs.get(option)
    if not isinstance(marks, dict):
        raise ValueError(f"Quality gate stopped the chain: expected criterion marks for {option}.")
    for criterion in criteria:
        if marks.get(criterion) not in ("+", "-"):
            raise ValueError(f"Quality gate stopped the chain: missing or invalid mark: {option} / {criterion}.")

## Step 5 · Call 3: record a decision and its cost
**Input:** the brief, `contexts`, and `tradeoffs`. **Output:** a short architecture decision record (ADR).

Ask for a context, decision, and consequence. The loop reads each line; a line without a colon is skipped, and the label check stops the chain when a field is missing. The following `if` statements check the labels and decision. Read the consequence yourself: the code cannot judge whether the cost is meaningful.

In [ ]:
prompt = (f"Brief: {brief}\nContexts: {contexts}\nTrade-offs: {json.dumps(tradeoffs)}\n"
          "Write exactly Context:, Decision:, Consequence: lines. Decision is one option label only. "
          "Consequence must name a cost; do not assume the service candidates are already deployed.")
record = ask(prompt)
print("Decision record:\n", record)
fields = {}
for line in record.strip().splitlines():
    if ": " in line:
        name, value = line.split(": ", 1)
        fields[name] = value
if list(fields) != ["Context", "Decision", "Consequence"]:
    raise ValueError("Quality gate stopped the chain: expected Context, Decision, Consequence lines. Rerun this cell to ask again.")
if fields["Decision"].strip().rstrip(".").lower() not in options:
    raise ValueError("Quality gate stopped the chain: Decision must be one of the two listed options.")

## What you should see
The lunch chain makes two model calls; the architecture chain makes three. Each output is printed so you can inspect what was passed to the next call. A failed gate stops the chain with a clear message — that message is the gate working, not the notebook breaking. Answers can vary. Gates check the expected structure, not the truth of every architectural claim.

## Try it
1. Make a gate fire on purpose: in Step 3, ask the model for exactly 4 labels while the gate still expects 3 (or change the gate to expect 4). Rerun Steps 3–5 and read the message. Why is a stop easier to debug than a wrong value quietly carried forward? Restore the cell before continuing.
2. Replace Order, Inventory, Shipping with UI, BusinessLogic, DataAccess. The list gate still passes: explain why the design may still be poor.
3. Explain why `Product` has different fields in Catalog, Inventory and Shipping when Catalog uses title and price, Inventory tracks quantity, and Shipping tracks package weight.
4. A customer returns a damaged jacket; after inspection it is refunded and then restocked or scrapped. Which language, rules and data would you investigate before drawing another boundary?

## Takeaway
Both examples pass one result to the next prompt. Chaining makes intermediate choices visible, and gates stop malformed handoffs where they start rather than where they surface later. Domain experts still validate proposed service boundaries.